In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install pandas scikit-learn pyarrow

# Dataset design - 1 seed

**IMPORTANT**: This notebook now saves as `.pkl` (pickle) instead of `.csv` to preserve full multi-dimensional embeddings.

The CSV format was truncating embeddings from 88,064 features to ~216 features due to numpy's string representation.
Parquet also cannot handle multi-dimensional arrays (only 1D), so pickle is used to preserve the original shape (8, 8, 1376).

In [ ]:
import numpy as np
import pandas as pd
from glob import glob
import os
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                            roc_auc_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

CONFIG = {
    'SAMPLE_SIZE': 2000,
    'TRAIN_TEST_SPLIT_RATIO': 0.8,
    'RANDOM_STATE': 42,

    'EMBEDDING_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/ve_cxr/elixir/batch_*.npz",
    'METADATA_PATH': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_clincalinfo.csv',
    'METADATA_EXTENDED_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_updatedlabel_wide_processed.csv",

    # Output directory - all results saved here
    'OUTPUT_DIR': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned',
    'OUTPUT_NAME': 'data_type5.pkl',  # Changed to pickle to preserve multi-dimensional arrays!

    'PATIENT_ID_COLUMN': 'subject_id',
    'GENDER_COLUMN': 'gender',
    'DICOM_PATH_COLUMN': 'dicom_path',

    'SAMPLING_STRATEGY': 5,

    'REMOVE_DUPLICATES': True,
    'REMOVE_DUPLICATE_FILENAMES': True,
    'ONE_IMAGE_PER_PATIENT': True,

    'STRIP_FILE_EXTENSION': True,
    'DEBUG_FILENAME_MATCHING': False,
    'VERBOSE': True
}

SAMPLING_STRATEGIES = {
    1: {
        'name': 'Balanced Gender (50/50)',
        'description': 'Ensures equal distribution across genders',
        'function': 'strategy_1_balanced_gender'
    },
    2: {
        'name': 'Stratified Proportional',
        'description': 'Maintains original gender proportions',
        'function': 'strategy_2_stratified_proportional'
    },
    3: {
        'name': 'Simple Random Sampling',
        'description': 'Pure random selection without stratification',
        'function': 'strategy_3_random_sampling'
    },
    4: {
        'name': 'Stratified by Pathology Status',
        'description': 'Preserves original distribution of normal vs abnormal cases',
        'function': 'strategy_4_pathology_stratified'
    },
    5: {
        'name': 'Stratified by Pathology Status (Excluding Support Devices Only)',
        'description': 'Preserves distribution, excludes cases with only Support Devices and No Finding',
        'function': 'strategy_5_pathology_stratified_clean'
    }
}

# Create output directory
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)

# Setup logging to file
log_messages = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"preprocessing_log_strategy{CONFIG['SAMPLING_STRATEGY']}_{timestamp}.txt"
log_path = os.path.join(CONFIG['OUTPUT_DIR'], log_filename)

def log_print(message):
    """Print message and store for later saving to file"""
    log_messages.append(message)
    if CONFIG['VERBOSE']:
        print(message)

def save_log():
    """Save all log messages to file"""
    with open(log_path, 'w') as f:
        f.write('\n'.join(log_messages))
    print(f"\nLog saved to: {log_path}")

log_print("=" * 80)
log_print("CHEST X-RAY GENDER CLASSIFICATION PIPELINE")
log_print("=" * 80)
log_print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_print(f"Target Sample Size: {CONFIG['SAMPLE_SIZE']:,} samples")
log_print(f"Train/Test split: {CONFIG['TRAIN_TEST_SPLIT_RATIO']:.0%} / {1-CONFIG['TRAIN_TEST_SPLIT_RATIO']:.0%}")
log_print(f"Sampling Strategy: {SAMPLING_STRATEGIES[CONFIG['SAMPLING_STRATEGY']]['name']}")
log_print(f"Output directory: {CONFIG['OUTPUT_DIR']}")
log_print(f"Output file: {CONFIG['OUTPUT_NAME']} (PICKLE format - preserves multi-dimensional embeddings!)")
log_print("=" * 80)

log_print("\n" + "=" * 80)
log_print("STEP 1: LOADING COMPLETE METADATA")
log_print("=" * 80)

meta = pd.read_csv(CONFIG['METADATA_PATH'])

log_print(f"Loaded metadata from: {CONFIG['METADATA_PATH']}")
log_print(f"TOTAL ROWS IN FULL DATASET: {len(meta):,}")
log_print(f"Original columns: {list(meta.columns)[:10]}...")

log_print("\nCreating filename column from dicom_path...")
meta['filename'] = meta[CONFIG['DICOM_PATH_COLUMN']].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)

if CONFIG['STRIP_FILE_EXTENSION']:
    meta['filename'] = meta['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)
    log_print("Stripped file extensions from filenames to match NPZ format")

meta = meta[meta['filename'].notna()].reset_index(drop=True)
log_print(f"Created filename column from {CONFIG['DICOM_PATH_COLUMN']}")
log_print(f"Rows after removing invalid paths: {len(meta):,}")
log_print(f"Example filename: {meta['filename'].iloc[0]}")

log_print(f"\nGender distribution in FULL dataset:")
gender_counts = meta[CONFIG['GENDER_COLUMN']].value_counts()
for gender, count in gender_counts.items():
    percentage = (count / len(meta)) * 100
    log_print(f"   {gender}: {count:,} ({percentage:.2f}%)")

unique_patients = meta[CONFIG['PATIENT_ID_COLUMN']].nunique()
log_print(f"\nTotal unique patients: {unique_patients:,}")
log_print(f"Average images per patient: {len(meta) / unique_patients:.2f}")

def remove_duplicates(df, patient_col, random_state):
    log_print("\n" + "=" * 80)
    log_print("STEP 2: REMOVING DUPLICATES - KEEPING ONLY UNIQUE DATA")
    log_print("=" * 80)

    original_size = len(df)
    log_print(f"Original dataset size: {original_size:,}")

    if CONFIG['REMOVE_DUPLICATE_FILENAMES']:
        log_print("\nRemoving duplicate filenames...")
        df_unique = df.drop_duplicates(subset=['filename'], keep='first')
        log_print(f"After removing duplicate filenames: {len(df_unique):,} rows")
        log_print(f"   Removed {original_size - len(df_unique):,} duplicate filenames")
    else:
        df_unique = df.copy()

    if CONFIG['ONE_IMAGE_PER_PATIENT']:
        log_print(f"\nKeeping only ONE image per patient (using {patient_col})...")
        patients_before = df_unique[patient_col].nunique()
        df_unique = df_unique.groupby(patient_col).sample(n=1, random_state=random_state).reset_index(drop=True)
        patients_after = df_unique[patient_col].nunique()

        log_print(f"Unique patients before: {patients_before:,}")
        log_print(f"Unique patients after: {patients_after:,}")
        log_print(f"Final unique dataset size: {len(df_unique):,} rows")
        log_print(f"Each row now represents a UNIQUE patient")

    log_print(f"\nDEDUPLICATION COMPLETE!")
    log_print(f"Reduced from {original_size:,} to {len(df_unique):,} UNIQUE samples")
    log_print(f"Reduction: {(1 - len(df_unique)/original_size)*100:.2f}%")

    log_print(f"\nGender distribution in unique dataset:")
    unique_gender = df_unique[CONFIG['GENDER_COLUMN']].value_counts()
    for gender, count in unique_gender.items():
        percentage = (count / len(df_unique)) * 100
        log_print(f"   {gender}: {count:,} ({percentage:.2f}%)")

    return df_unique

if CONFIG['REMOVE_DUPLICATES']:
    meta_unique = remove_duplicates(meta, CONFIG['PATIENT_ID_COLUMN'], CONFIG['RANDOM_STATE'])
else:
    meta_unique = meta.copy()
    log_print("\nSkipping deduplication (REMOVE_DUPLICATES=False)")

def strategy_1_balanced_gender(df, target_size, random_state, gender_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 1 - BALANCED GENDER (50/50)")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples ({target_size//2:,} per gender)")
    log_print(f"Available: {len(df):,} unique samples")

    df_clean = df[df[gender_col].notna()].copy()
    log_print(f"Removed {len(df) - len(df_clean):,} rows with missing gender values")
    log_print(f"Clean dataset: {len(df_clean):,} samples")

    samples_per_gender = target_size // 2
    sampled_dfs = []

    for gender_value in sorted(df_clean[gender_col].unique()):
        log_print(f"\nSampling gender: {gender_value}")
        gender_df = df_clean[df_clean[gender_col] == gender_value]
        log_print(f"   Available {gender_value}: {len(gender_df):,}")

        if len(gender_df) >= samples_per_gender:
            gender_sample = gender_df.sample(n=samples_per_gender, random_state=random_state)
            log_print(f"   Sampled {samples_per_gender:,} for {gender_value}")
        else:
            gender_sample = gender_df
            log_print(f"   Only {len(gender_sample):,} available for {gender_value}")

        sampled_dfs.append(gender_sample)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Gender distribution:")
    for gender, count in df_sampled[gender_col].value_counts().items():
        log_print(f"   {gender}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

def strategy_2_stratified_proportional(df, target_size, random_state, gender_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 2 - STRATIFIED PROPORTIONAL")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples")
    log_print(f"Available: {len(df):,} unique samples")
    log_print(f"Maintains original gender proportions")

    df_clean = df[df[gender_col].notna()].copy()
    log_print(f"Removed {len(df) - len(df_clean):,} rows with missing gender values")
    log_print(f"Clean dataset: {len(df_clean):,} samples")

    gender_proportions = df_clean[gender_col].value_counts(normalize=True)
    log_print(f"\nOriginal proportions:")
    for gender, prop in gender_proportions.items():
        log_print(f"   {gender}: {prop*100:.2f}%")

    # Sample from each gender group proportionally (using concat instead of groupby.apply)
    sampled_dfs = []
    for gender_value in df_clean[gender_col].unique():
        gender_df = df_clean[df_clean[gender_col] == gender_value]
        n_samples = int(target_size * len(gender_df) / len(df_clean))
        if len(gender_df) >= n_samples:
            sampled_dfs.append(gender_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(gender_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Final gender distribution:")
    for gender, count in df_sampled[gender_col].value_counts().items():
        log_print(f"   {gender}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

def strategy_3_random_sampling(df, target_size, random_state, gender_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 3 - SIMPLE RANDOM SAMPLING")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples")
    log_print(f"Available: {len(df):,} unique samples")
    log_print(f"Pure random selection (no stratification)")

    df_clean = df[df[gender_col].notna()].copy()
    log_print(f"Removed {len(df) - len(df_clean):,} rows with missing gender values")
    log_print(f"Clean dataset: {len(df_clean):,} samples")

    if len(df_clean) >= target_size:
        df_sampled = df_clean.sample(n=target_size, random_state=random_state).reset_index(drop=True)
        log_print(f"\nSampled {target_size:,} random samples")
    else:
        df_sampled = df_clean
        log_print(f"\nOnly {len(df_clean):,} samples available")

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Gender distribution:")
    for gender, count in df_sampled[gender_col].value_counts().items():
        log_print(f"   {gender}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

def strategy_4_pathology_stratified(df, target_size, random_state, gender_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 4 - STRATIFIED BY PATHOLOGY STATUS")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples")
    log_print(f"Available: {len(df):,} unique samples")
    log_print(f"Preserves original distribution of normal vs abnormal cases")

    log_print("\nLoading extended metadata with pathology labels...")
    meta_extended = pd.read_csv(CONFIG['METADATA_EXTENDED_PATH'])

    log_print(f"Extended metadata loaded: {len(meta_extended):,} rows")
    log_print(f"Pathology columns available: {[col for col in meta_extended.columns if col not in ['dicom_path', 'dicom_id', 'subject_id', 'study_id', 'gender']]}")

    meta_extended['filename'] = meta_extended['dicom_path'].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)
    if CONFIG['STRIP_FILE_EXTENSION']:
        meta_extended['filename'] = meta_extended['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)

    df_merged = pd.merge(df, meta_extended[['filename', 'No Finding', 'Atelectasis', 'Cardiomegaly',
                                             'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
                                             'Fracture', 'Lung Lesion', 'Lung Opacity',
                                             'Pleural Effusion', 'Pleural Other', 'Pneumonia',
                                             'Pneumothorax', 'Support Devices']],
                        on='filename', how='inner')

    log_print(f"After merging with pathology data: {len(df_merged):,} rows")

    pathology_cols = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                     'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                     'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
                     'Pneumonia', 'Pneumothorax', 'Support Devices']

    df_merged['has_pathology'] = df_merged[pathology_cols].fillna(0).sum(axis=1) > 0
    df_merged['pathology_status'] = df_merged['has_pathology'].apply(lambda x: 'abnormal' if x else 'normal')

    log_print(f"\nPathology status distribution in available data:")
    status_counts = df_merged['pathology_status'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / len(df_merged)) * 100
        log_print(f"   {status}: {count:,} ({percentage:.2f}%)")

    df_clean = df_merged[df_merged[gender_col].notna()].copy()
    log_print(f"Removed {len(df_merged) - len(df_clean):,} rows with missing gender values")
    log_print(f"Clean dataset: {len(df_clean):,} samples")

    status_proportions = df_clean['pathology_status'].value_counts(normalize=True)
    log_print(f"\nOriginal pathology proportions to preserve:")
    for status, prop in status_proportions.items():
        log_print(f"   {status}: {prop*100:.2f}%")

    # Sample from each pathology status group proportionally (using concat instead of groupby.apply)
    sampled_dfs = []
    for status in df_clean['pathology_status'].unique():
        status_df = df_clean[df_clean['pathology_status'] == status]
        n_samples = int(target_size * len(status_df) / len(df_clean))
        if len(status_df) >= n_samples:
            sampled_dfs.append(status_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(status_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Final pathology status distribution:")
    for status, count in df_sampled['pathology_status'].value_counts().items():
        log_print(f"   {status}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    log_print(f"\nFinal gender distribution:")
    for gender, count in df_sampled[gender_col].value_counts().items():
        log_print(f"   {gender}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

def strategy_5_pathology_stratified_clean(df, target_size, random_state, gender_col):
    log_print("\n" + "=" * 80)
    log_print(f"STEP 3: SAMPLING STRATEGY 5 - STRATIFIED BY PATHOLOGY (CLEAN)")
    log_print("=" * 80)
    log_print(f"Target: {target_size:,} samples")
    log_print(f"Available: {len(df):,} unique samples")
    log_print(f"Preserves distribution, excludes cases with only Support Devices + No Finding")

    log_print("\nLoading extended metadata with pathology labels...")
    meta_extended = pd.read_csv(CONFIG['METADATA_EXTENDED_PATH'])

    log_print(f"Extended metadata loaded: {len(meta_extended):,} rows")

    meta_extended['filename'] = meta_extended['dicom_path'].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)
    if CONFIG['STRIP_FILE_EXTENSION']:
        meta_extended['filename'] = meta_extended['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)

    df_merged = pd.merge(df, meta_extended[['filename', 'No Finding', 'Atelectasis', 'Cardiomegaly',
                                             'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
                                             'Fracture', 'Lung Lesion', 'Lung Opacity',
                                             'Pleural Effusion', 'Pleural Other', 'Pneumonia',
                                             'Pneumothorax', 'Support Devices']],
                        on='filename', how='inner')

    log_print(f"After merging with pathology data: {len(df_merged):,} rows")

    pathology_cols_no_support = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                                  'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                                  'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
                                  'Pneumonia', 'Pneumothorax']

    df_merged['has_real_pathology'] = df_merged[pathology_cols_no_support].fillna(0).sum(axis=1) > 0
    df_merged['has_support_devices'] = df_merged['Support Devices'].fillna(0) > 0
    df_merged['has_no_finding'] = df_merged['No Finding'].fillna(0) > 0

    support_only_mask = (df_merged['has_support_devices']) & (df_merged['has_no_finding']) & (~df_merged['has_real_pathology'])

    log_print(f"\nExcluding {support_only_mask.sum():,} cases with only Support Devices + No Finding")
    df_filtered = df_merged[~support_only_mask].copy()
    log_print(f"Dataset after exclusion: {len(df_filtered):,} rows")

    df_filtered['pathology_status'] = df_filtered['has_real_pathology'].apply(lambda x: 'abnormal' if x else 'normal')

    log_print(f"\nPathology status distribution in filtered data:")
    status_counts = df_filtered['pathology_status'].value_counts()
    for status, count in status_counts.items():
        percentage = (count / len(df_filtered)) * 100
        log_print(f"   {status}: {count:,} ({percentage:.2f}%)")

    df_clean = df_filtered[df_filtered[gender_col].notna()].copy()
    log_print(f"Removed {len(df_filtered) - len(df_clean):,} rows with missing gender values")
    log_print(f"Clean dataset: {len(df_clean):,} samples")

    status_proportions = df_clean['pathology_status'].value_counts(normalize=True)
    log_print(f"\nOriginal pathology proportions to preserve:")
    for status, prop in status_proportions.items():
        log_print(f"   {status}: {prop*100:.2f}%")

    # Sample from each pathology status group proportionally (using concat instead of groupby.apply)
    sampled_dfs = []
    for status in df_clean['pathology_status'].unique():
        status_df = df_clean[df_clean['pathology_status'] == status]
        n_samples = int(target_size * len(status_df) / len(df_clean))
        if len(status_df) >= n_samples:
            sampled_dfs.append(status_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(status_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    log_print(f"\nFinal sample size: {len(df_sampled):,}")
    log_print(f"Final pathology status distribution:")
    for status, count in df_sampled['pathology_status'].value_counts().items():
        log_print(f"   {status}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    log_print(f"\nFinal gender distribution:")
    for gender, count in df_sampled[gender_col].value_counts().items():
        log_print(f"   {gender}: {count:,} ({count/len(df_sampled)*100:.2f}%)")

    return df_sampled

strategy = CONFIG['SAMPLING_STRATEGY']
if strategy == 1:
    meta_sampled = strategy_1_balanced_gender(meta_unique, CONFIG['SAMPLE_SIZE'],
                                              CONFIG['RANDOM_STATE'], CONFIG['GENDER_COLUMN'])
elif strategy == 2:
    meta_sampled = strategy_2_stratified_proportional(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                      CONFIG['RANDOM_STATE'], CONFIG['GENDER_COLUMN'])
elif strategy == 3:
    meta_sampled = strategy_3_random_sampling(meta_unique, CONFIG['SAMPLE_SIZE'],
                                              CONFIG['RANDOM_STATE'], CONFIG['GENDER_COLUMN'])
elif strategy == 4:
    meta_sampled = strategy_4_pathology_stratified(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                   CONFIG['RANDOM_STATE'], CONFIG['GENDER_COLUMN'])
elif strategy == 5:
    meta_sampled = strategy_5_pathology_stratified_clean(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                         CONFIG['RANDOM_STATE'], CONFIG['GENDER_COLUMN'])
else:
    raise ValueError(f"Invalid SAMPLING_STRATEGY: {strategy}. Must be 1, 2, 3, 4, or 5.")

log_print("\n" + "=" * 80)
log_print(f"STEP 4: LOADING ONLY {len(meta_sampled):,} REQUIRED EMBEDDINGS")
log_print("=" * 80)

filenames_to_load = set(meta_sampled['filename'].values)
log_print(f"Created lookup set with {len(filenames_to_load):,} filenames")
log_print(f"Example filename to match: {list(filenames_to_load)[0]}")

if CONFIG['DEBUG_FILENAME_MATCHING']:
    log_print("\n" + "=" * 80)
    log_print("DEBUG: FILENAME FORMAT ANALYSIS")
    log_print("=" * 80)

    sample_metadata_filenames = list(filenames_to_load)[:5]
    log_print(f"\nSample metadata filenames (first 5):")
    for i, fname in enumerate(sample_metadata_filenames, 1):
        log_print(f"  {i}. {fname}")

    batch_files = sorted(glob(CONFIG['EMBEDDING_PATH']))
    if len(batch_files) > 0:
        first_batch = np.load(batch_files[0], allow_pickle=True)
        sample_npz_filenames = first_batch["filenames"][:5]
        log_print(f"\nSample NPZ filenames (first 5 from first batch):")
        for i, fname in enumerate(sample_npz_filenames, 1):
            log_print(f"  {i}. {fname}")

        log_print(f"\nFilename format comparison:")
        log_print(f"  Metadata filename type: {type(sample_metadata_filenames[0])}")
        log_print(f"  NPZ filename type: {type(sample_npz_filenames[0])}")

    log_print("=" * 80)

log_print("\nOnly loading embeddings we need - this is FAST!")

records = []
batch_files = sorted(glob(CONFIG['EMBEDDING_PATH']))
log_print(f"\nFound {len(batch_files)} batch files")

embeddings_found = 0

for path in tqdm(batch_files, desc="Scanning batches", disable=not CONFIG['VERBOSE']):
    batch_name = os.path.basename(path)
    data = np.load(path, allow_pickle=True)

    for emb, fname in zip(data["embeddings"], data["filenames"]):
        fname_clean = str(fname)

        if fname_clean in filenames_to_load:
            records.append({
                "embedding": emb,  # Keep original shape (8, 8, 1376)!
                "filename": fname_clean,
                "batch": batch_name
            })
            embeddings_found += 1

            if embeddings_found >= len(filenames_to_load):
                break

    if embeddings_found >= len(filenames_to_load):
        log_print(f"\nFound all {len(filenames_to_load):,} embeddings! Stopping early.")
        break

log_print(f"\nEMBEDDINGS LOADED: {len(records):,}")

if len(records) == 0:
    log_print("\nERROR: No embeddings matched!")
    log_print("This indicates a filename format mismatch.")
    log_print("\nTroubleshooting:")
    log_print(f"  STRIP_FILE_EXTENSION is set to: {CONFIG['STRIP_FILE_EXTENSION']}")
    log_print(f"  Example metadata filename: {list(filenames_to_load)[0]}")
    log_print("\nEnable DEBUG_FILENAME_MATCHING=True in CONFIG to see detailed comparison.")
    save_log()  # Save log before raising error
    raise ValueError("No embeddings matched. Enable DEBUG_FILENAME_MATCHING for more details.")

emb_df = pd.DataFrame(records)

log_print(f"Match rate: {len(emb_df):,} / {len(filenames_to_load):,} ({len(emb_df)/len(filenames_to_load)*100:.2f}%)")

if len(records) > 0:
    log_print(f"Embedding shape: {records[0]['embedding'].shape}")
    log_print(f"Embedding dimension (flattened): {records[0]['embedding'].flatten().shape[0]:,} features")

log_print("\n" + "=" * 80)
log_print("STEP 5: MERGING SAMPLED METADATA WITH EMBEDDINGS")
log_print("=" * 80)

df_final = pd.merge(meta_sampled, emb_df, on="filename", how="inner")

log_print(f"FINAL DATASET SIZE: {len(df_final):,} rows")
log_print(f"This is your {CONFIG['SAMPLE_SIZE']:,} sample dataset ready for training!")

log_print(f"\nFinal gender distribution:")
for gender, count in df_final[CONFIG['GENDER_COLUMN']].value_counts().items():
    percentage = (count / len(df_final)) * 100
    log_print(f"   {gender}: {count:,} ({percentage:.2f}%)")

log_print(f"\nAvailable columns in final dataset: {list(df_final.columns)[:15]}...")

# Use OUTPUT_DIR for saving
output_path = os.path.join(CONFIG['OUTPUT_DIR'], CONFIG['OUTPUT_NAME'])

log_print("\n" + "=" * 80)
log_print("STEP 6: SAVING OUTPUT (PICKLE FORMAT)")
log_print("=" * 80)

# Save as pickle to preserve multi-dimensional numpy arrays!
df_final.to_pickle(output_path)

log_print(f"Output saved to: {output_path}")
log_print(f"Total rows saved: {len(df_final):,}")
log_print(f"File size: {os.path.getsize(output_path) / (1024*1024):.2f} MB")

# Verify the embedding was saved correctly
log_print("\n" + "=" * 80)
log_print("VERIFICATION: Checking saved embeddings")
log_print("=" * 80)
df_verify = pd.read_pickle(output_path)
log_print(f"Loaded back {len(df_verify):,} rows")
log_print(f"Embedding shape after reload: {df_verify['embedding'].iloc[0].shape}")
log_print(f"Embedding size after reload: {df_verify['embedding'].iloc[0].flatten().shape[0]:,} features")
log_print(f"First 5 values: {df_verify['embedding'].iloc[0].flatten()[:5]}")

log_print("\n" + "=" * 80)
log_print("PIPELINE COMPLETE!")
log_print("=" * 80)
log_print(f"\nFull embeddings preserved: {df_verify['embedding'].iloc[0].shape} = {df_verify['embedding'].iloc[0].flatten().shape[0]:,} features")
log_print(f"\nOutput files saved to: {CONFIG['OUTPUT_DIR']}")
log_print(f"  - Data: {CONFIG['OUTPUT_NAME']}")
log_print(f"  - Log: {log_filename}")

# Save log file
save_log()

CHEST X-RAY GENDER CLASSIFICATION PIPELINE
Timestamp: 2025-12-12 15:57:11
Target Sample Size: 2,000 samples
Train/Test split: 80% / 20%
Sampling Strategy: Stratified by Pathology Status (Excluding Support Devices Only)
Output directory: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned
Output file: data_type5.pkl (PICKLE format - preserves multi-dimensional embeddings!)

STEP 1: LOADING COMPLETE METADATA
Loaded metadata from: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_clincalinfo.csv
TOTAL ROWS IN FULL DATASET: 243,334
Original columns: ['dicom_path', 'dicom_id', 'subject_id', 'study_id', 'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns', 'ProcedureCodeSequence_CodeMeaning', 'ViewCodeSequence_CodeMeaning']...

Creating filename column from dicom_path...
Stripped file extensions from filenames to match NPZ format
Created filename column from dicom_path
Rows after removing invalid paths: 243,334
Example filenam

Scanning batches:  98%|█████████▊| 48/49 [26:17<00:32, 32.86s/it]


Found all 1,999 embeddings! Stopping early.

EMBEDDINGS LOADED: 1,999
Match rate: 1,999 / 1,999 (100.00%)
Embedding shape: (8, 8, 1376)
Embedding dimension (flattened): 88,064 features

STEP 5: MERGING SAMPLED METADATA WITH EMBEDDINGS


FINAL DATASET SIZE: 1,999 rows
This is your 2,000 sample dataset ready for training!

Final gender distribution:
   F: 1,054 (52.73%)
   M: 945 (47.27%)

Available columns in final dataset: ['dicom_path', 'dicom_id', 'subject_id', 'study_id', 'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns', 'ProcedureCodeSequence_CodeMeaning', 'ViewCodeSequence_CodeMeaning', 'PatientOrientationCodeSequence_CodeMeaning', 'gender', 'anchor_age', 'anchor_year', 'anchor_year_group']...

STEP 6: SAVING OUTPUT (PICKLE FORMAT)
Output saved to: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned/data_type5.pkl
Total rows saved: 1,999
File size: 673.05 MB

VERIFICATION: Checking saved embeddings
Loaded back 1,999 rows
Embedding shape after reload: (8, 8, 1376)
Embedding size after reload: 88,064 features
First 5 values: [-5.4925885 -5.875949  -1.8365123 -8.202694   2.1226118]

PIPELINE COMPLETE!

Full embeddings preserved: (8, 8, 1376) = 88,064 features

Output files

This notebook implements 5 sampling strategies for gender classification. For detailed descriptions of each strategy, see the **Sampling Strategies** section in the main [README](../README.md).

To use different strategies, change `'SAMPLING_STRATEGY'` in the CONFIG to the number you want (1-5), and update `'OUTPUT_NAME'` accordingly (e.g., `'data_type4.pkl'` or `'data_type5.pkl'`).

# Dataset design - multiple seeds



1. **Added `NUM_SEEDS` parameter** in CONFIG (set to 20)
2. **Created output folder structure** based on sampling strategy name (e.g., "data_type1", "data_type4", etc.)
3. **Load all embeddings once** at the beginning instead of loading them 20 times
4. **Loop through seeds** from 1 to 20, generating different samples for each seed
5. **Each seed gets its own file**: `data_type1_seed1.pkl`, `data_type1_seed2.pkl`, etc.
6. **Different random states** for each seed: BASE_RANDOM_STATE + seed_idx (so 43, 44, 45... up to 62)

To use it, just change `SAMPLING_STRATEGY` in CONFIG to the desired strategy (1-5), and it will automatically create the appropriate folder and 20 seed files.

**IMPORTANT**: Now saves as `.pkl` (pickle) instead of `.csv` to preserve full multi-dimensional embeddings (shape 8, 8, 1376).

In [3]:
import numpy as np
import pandas as pd
from glob import glob
import os
from tqdm import tqdm
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                            roc_auc_score, confusion_matrix, classification_report)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns
import gc
from datetime import datetime

CONFIG = {
    'SAMPLE_SIZE': 2000,
    'TRAIN_TEST_SPLIT_RATIO': 0.8,
    'BASE_RANDOM_STATE': 42,
    'NUM_SEEDS': 20,

    'EMBEDDING_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/ve_cxr/elixir/batch_*.npz",
    'METADATA_PATH': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_clincalinfo.csv',
    'METADATA_EXTENDED_PATH': "/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_updatedlabel_wide_processed.csv",

    # Output directory - all results saved here in data-cleaned/data_typeX/
    'OUTPUT_BASE_DIR': '/content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned-seeding',

    'OUTPUT_NAME': 'data_type1.pkl',  # Changed to pickle to preserve multi-dimensional arrays!

    'PATIENT_ID_COLUMN': 'subject_id',
    'GENDER_COLUMN': 'gender',
    'DICOM_PATH_COLUMN': 'dicom_path',

    'SAMPLING_STRATEGY': 1,

    'REMOVE_DUPLICATES': True,
    'REMOVE_DUPLICATE_FILENAMES': True,
    'ONE_IMAGE_PER_PATIENT': True,

    'STRIP_FILE_EXTENSION': True,
    'DEBUG_FILENAME_MATCHING': False,
    'VERBOSE': True
}

SAMPLING_STRATEGIES = {
    1: {
        'name': 'Balanced Gender (50/50)',
        'description': 'Ensures equal distribution across genders',
        'function': 'strategy_1_balanced_gender',
        'folder_name': 'data_type1'
    },
    2: {
        'name': 'Stratified Proportional',
        'description': 'Maintains original gender proportions',
        'function': 'strategy_2_stratified_proportional',
        'folder_name': 'data_type2'
    },
    3: {
        'name': 'Simple Random Sampling',
        'description': 'Pure random selection without stratification',
        'function': 'strategy_3_random_sampling',
        'folder_name': 'data_type3'
    },
    4: {
        'name': 'Stratified by Pathology Status',
        'description': 'Preserves original distribution of normal vs abnormal cases',
        'function': 'strategy_4_pathology_stratified',
        'folder_name': 'data_type4'
    },
    5: {
        'name': 'Stratified by Pathology Status (Excluding Support Devices Only)',
        'description': 'Preserves distribution, excludes cases with only Support Devices and No Finding',
        'function': 'strategy_5_pathology_stratified_clean',
        'folder_name': 'data_type5'
    }
}

# Create output directories
output_folder = SAMPLING_STRATEGIES[CONFIG['SAMPLING_STRATEGY']]['folder_name']
output_dir = os.path.join(CONFIG['OUTPUT_BASE_DIR'], output_folder)
os.makedirs(output_dir, exist_ok=True)

# Setup logging to file
log_messages = []
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
log_filename = f"preprocessing_log_multipleseeds_{output_folder}_{timestamp}.txt"
log_path = os.path.join(output_dir, log_filename)

def log_print(message, end='\n'):
    """Print message and store for later saving to file"""
    log_messages.append(message)
    if CONFIG['VERBOSE']:
        print(message, end=end)

def save_log():
    """Save all log messages to file"""
    with open(log_path, 'w') as f:
        f.write('\n'.join(log_messages))
    print(f"\nLog saved to: {log_path}")

log_print("=" * 80)
log_print("CHEST X-RAY GENDER CLASSIFICATION PIPELINE - MULTIPLE SEEDS")
log_print("=" * 80)
log_print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
log_print(f"Target Sample Size: {CONFIG['SAMPLE_SIZE']:,} samples")
log_print(f"Number of Seeds: {CONFIG['NUM_SEEDS']}")
log_print(f"Base Random State: {CONFIG['BASE_RANDOM_STATE']}")
log_print(f"Train/Test split: {CONFIG['TRAIN_TEST_SPLIT_RATIO']:.0%} / {1-CONFIG['TRAIN_TEST_SPLIT_RATIO']:.0%}")
log_print(f"Sampling Strategy: {SAMPLING_STRATEGIES[CONFIG['SAMPLING_STRATEGY']]['name']}")
log_print(f"Output Directory: {output_dir}")
log_print(f"Output Format: PICKLE (preserves multi-dimensional embeddings!)")
log_print("=" * 80)

log_print("\n" + "=" * 80)
log_print("STEP 1: LOADING COMPLETE METADATA")
log_print("=" * 80)

meta = pd.read_csv(CONFIG['METADATA_PATH'])

log_print(f"Loaded metadata from: {CONFIG['METADATA_PATH']}")
log_print(f"TOTAL ROWS IN FULL DATASET: {len(meta):,}")
log_print(f"Original columns: {list(meta.columns)[:10]}...")

log_print("\nCreating filename column from dicom_path...")
meta['filename'] = meta[CONFIG['DICOM_PATH_COLUMN']].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)

if CONFIG['STRIP_FILE_EXTENSION']:
    meta['filename'] = meta['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)
    log_print("Stripped file extensions from filenames to match NPZ format")

meta = meta[meta['filename'].notna()].reset_index(drop=True)
log_print(f"Created filename column from {CONFIG['DICOM_PATH_COLUMN']}")
log_print(f"Rows after removing invalid paths: {len(meta):,}")
log_print(f"Example filename: {meta['filename'].iloc[0]}")

log_print(f"\nGender distribution in FULL dataset:")
gender_counts = meta[CONFIG['GENDER_COLUMN']].value_counts()
for gender, count in gender_counts.items():
    percentage = (count / len(meta)) * 100
    log_print(f"   {gender}: {count:,} ({percentage:.2f}%)")

unique_patients = meta[CONFIG['PATIENT_ID_COLUMN']].nunique()
log_print(f"\nTotal unique patients: {unique_patients:,}")
log_print(f"Average images per patient: {len(meta) / unique_patients:.2f}")

def remove_duplicates(df, patient_col, random_state):
    log_print("\n" + "=" * 80)
    log_print("STEP 2: REMOVING DUPLICATES - KEEPING ONLY UNIQUE DATA")
    log_print("=" * 80)

    original_size = len(df)
    log_print(f"Original dataset size: {original_size:,}")

    if CONFIG['REMOVE_DUPLICATE_FILENAMES']:
        log_print("\nRemoving duplicate filenames...")
        df_unique = df.drop_duplicates(subset=['filename'], keep='first')
        log_print(f"After removing duplicate filenames: {len(df_unique):,} rows")
        log_print(f"   Removed {original_size - len(df_unique):,} duplicate filenames")
    else:
        df_unique = df.copy()

    if CONFIG['ONE_IMAGE_PER_PATIENT']:
        log_print(f"\nKeeping only ONE image per patient (using {patient_col})...")
        patients_before = df_unique[patient_col].nunique()
        df_unique = df_unique.groupby(patient_col).sample(n=1, random_state=random_state).reset_index(drop=True)
        patients_after = df_unique[patient_col].nunique()

        log_print(f"Unique patients before: {patients_before:,}")
        log_print(f"Unique patients after: {patients_after:,}")
        log_print(f"Final unique dataset size: {len(df_unique):,} rows")
        log_print(f"Each row now represents a UNIQUE patient")

    log_print(f"\nDEDUPLICATION COMPLETE!")
    log_print(f"Reduced from {original_size:,} to {len(df_unique):,} UNIQUE samples")
    log_print(f"Reduction: {(1 - len(df_unique)/original_size)*100:.2f}%")

    log_print(f"\nGender distribution in unique dataset:")
    unique_gender = df_unique[CONFIG['GENDER_COLUMN']].value_counts()
    for gender, count in unique_gender.items():
        percentage = (count / len(df_unique)) * 100
        log_print(f"   {gender}: {count:,} ({percentage:.2f}%)")

    return df_unique

if CONFIG['REMOVE_DUPLICATES']:
    meta_unique = remove_duplicates(meta, CONFIG['PATIENT_ID_COLUMN'], CONFIG['BASE_RANDOM_STATE'])
else:
    meta_unique = meta.copy()
    log_print("\nSkipping deduplication (REMOVE_DUPLICATES=False)")

def strategy_1_balanced_gender(df, target_size, random_state, gender_col):
    log_print(f"\nSTRATEGY 1 - BALANCED GENDER (50/50) - Seed {random_state}")
    log_print(f"Target: {target_size:,} samples ({target_size//2:,} per gender)")

    df_clean = df[df[gender_col].notna()].copy()

    samples_per_gender = target_size // 2
    sampled_dfs = []

    for gender_value in sorted(df_clean[gender_col].unique()):
        gender_df = df_clean[df_clean[gender_col] == gender_value]

        if len(gender_df) >= samples_per_gender:
            gender_sample = gender_df.sample(n=samples_per_gender, random_state=random_state)
        else:
            gender_sample = gender_df

        sampled_dfs.append(gender_sample)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    gender_dist = " | ".join([f"{gender}={count:,}" for gender, count in df_sampled[gender_col].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({gender_dist})")

    return df_sampled

def strategy_2_stratified_proportional(df, target_size, random_state, gender_col):
    log_print(f"\nSTRATEGY 2 - STRATIFIED PROPORTIONAL - Seed {random_state}")
    log_print(f"Target: {target_size:,} samples")

    df_clean = df[df[gender_col].notna()].copy()

    # Sample from each gender group proportionally (using concat instead of groupby.apply)
    sampled_dfs = []
    for gender_value in df_clean[gender_col].unique():
        gender_df = df_clean[df_clean[gender_col] == gender_value]
        n_samples = int(target_size * len(gender_df) / len(df_clean))
        if len(gender_df) >= n_samples:
            sampled_dfs.append(gender_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(gender_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    gender_dist = " | ".join([f"{gender}={count:,}" for gender, count in df_sampled[gender_col].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({gender_dist})")

    return df_sampled

def strategy_3_random_sampling(df, target_size, random_state, gender_col):
    log_print(f"\nSTRATEGY 3 - SIMPLE RANDOM SAMPLING - Seed {random_state}")
    log_print(f"Target: {target_size:,} samples")

    df_clean = df[df[gender_col].notna()].copy()

    if len(df_clean) >= target_size:
        df_sampled = df_clean.sample(n=target_size, random_state=random_state).reset_index(drop=True)
    else:
        df_sampled = df_clean

    gender_dist = " | ".join([f"{gender}={count:,}" for gender, count in df_sampled[gender_col].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({gender_dist})")

    return df_sampled

def strategy_4_pathology_stratified(df, target_size, random_state, gender_col):
    log_print(f"\nSTRATEGY 4 - STRATIFIED BY PATHOLOGY STATUS - Seed {random_state}")
    log_print(f"Target: {target_size:,} samples")

    meta_extended = pd.read_csv(CONFIG['METADATA_EXTENDED_PATH'])

    meta_extended['filename'] = meta_extended['dicom_path'].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)
    if CONFIG['STRIP_FILE_EXTENSION']:
        meta_extended['filename'] = meta_extended['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)

    df_merged = pd.merge(df, meta_extended[['filename', 'No Finding', 'Atelectasis', 'Cardiomegaly',
                                             'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
                                             'Fracture', 'Lung Lesion', 'Lung Opacity',
                                             'Pleural Effusion', 'Pleural Other', 'Pneumonia',
                                             'Pneumothorax', 'Support Devices']],
                        on='filename', how='inner')

    pathology_cols = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                     'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                     'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
                     'Pneumonia', 'Pneumothorax', 'Support Devices']

    df_merged['has_pathology'] = df_merged[pathology_cols].fillna(0).sum(axis=1) > 0
    df_merged['pathology_status'] = df_merged['has_pathology'].apply(lambda x: 'abnormal' if x else 'normal')

    df_clean = df_merged[df_merged[gender_col].notna()].copy()

    # Sample from each pathology status group proportionally (using concat instead of groupby.apply)
    sampled_dfs = []
    for status in df_clean['pathology_status'].unique():
        status_df = df_clean[df_clean['pathology_status'] == status]
        n_samples = int(target_size * len(status_df) / len(df_clean))
        if len(status_df) >= n_samples:
            sampled_dfs.append(status_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(status_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    path_dist = " | ".join([f"{status}={count:,}" for status, count in df_sampled['pathology_status'].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({path_dist})")

    return df_sampled

def strategy_5_pathology_stratified_clean(df, target_size, random_state, gender_col):
    log_print(f"\nSTRATEGY 5 - STRATIFIED BY PATHOLOGY (CLEAN) - Seed {random_state}")
    log_print(f"Target: {target_size:,} samples")

    meta_extended = pd.read_csv(CONFIG['METADATA_EXTENDED_PATH'])

    meta_extended['filename'] = meta_extended['dicom_path'].apply(lambda x: os.path.basename(x) if pd.notna(x) else None)
    if CONFIG['STRIP_FILE_EXTENSION']:
        meta_extended['filename'] = meta_extended['filename'].apply(lambda x: os.path.splitext(x)[0] if pd.notna(x) else None)

    df_merged = pd.merge(df, meta_extended[['filename', 'No Finding', 'Atelectasis', 'Cardiomegaly',
                                             'Consolidation', 'Edema', 'Enlarged Cardiomediastinum',
                                             'Fracture', 'Lung Lesion', 'Lung Opacity',
                                             'Pleural Effusion', 'Pleural Other', 'Pneumonia',
                                             'Pneumothorax', 'Support Devices']],
                        on='filename', how='inner')

    pathology_cols_no_support = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
                                  'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
                                  'Lung Opacity', 'Pleural Effusion', 'Pleural Other',
                                  'Pneumonia', 'Pneumothorax']

    df_merged['has_real_pathology'] = df_merged[pathology_cols_no_support].fillna(0).sum(axis=1) > 0
    df_merged['has_support_devices'] = df_merged['Support Devices'].fillna(0) > 0
    df_merged['has_no_finding'] = df_merged['No Finding'].fillna(0) > 0

    support_only_mask = (df_merged['has_support_devices']) & (df_merged['has_no_finding']) & (~df_merged['has_real_pathology'])

    df_filtered = df_merged[~support_only_mask].copy()

    df_filtered['pathology_status'] = df_filtered['has_real_pathology'].apply(lambda x: 'abnormal' if x else 'normal')

    df_clean = df_filtered[df_filtered[gender_col].notna()].copy()

    # Sample from each pathology status group proportionally (using concat instead of groupby.apply)
    sampled_dfs = []
    for status in df_clean['pathology_status'].unique():
        status_df = df_clean[df_clean['pathology_status'] == status]
        n_samples = int(target_size * len(status_df) / len(df_clean))
        if len(status_df) >= n_samples:
            sampled_dfs.append(status_df.sample(n=n_samples, random_state=random_state))
        else:
            sampled_dfs.append(status_df)

    df_sampled = pd.concat(sampled_dfs, ignore_index=True)

    path_dist = " | ".join([f"{status}={count:,}" for status, count in df_sampled['pathology_status'].value_counts().items()])
    log_print(f"Sample size: {len(df_sampled):,} ({path_dist})")

    return df_sampled

def load_embeddings_for_filenames(filenames_set):
    """Load only the embeddings needed for the given filenames"""
    records = []
    batch_files = sorted(glob(CONFIG['EMBEDDING_PATH']))
    embeddings_found = 0

    for path in tqdm(batch_files, desc="Loading embeddings", disable=not CONFIG['VERBOSE']):
        data = np.load(path, allow_pickle=True)

        for emb, fname in zip(data["embeddings"], data["filenames"]):
            fname_clean = str(fname)

            if fname_clean in filenames_set:
                records.append({
                    "embedding": emb,  # Keep original shape!
                    "filename": fname_clean
                })
                embeddings_found += 1

                if embeddings_found >= len(filenames_set):
                    break

        if embeddings_found >= len(filenames_set):
            break

    return pd.DataFrame(records)

log_print("\n" + "=" * 80)
log_print(f"Output directory created: {output_dir}")
log_print("=" * 80)

log_print("\n" + "=" * 80)
log_print(f"GENERATING {CONFIG['NUM_SEEDS']} DIFFERENT SAMPLES")
log_print("=" * 80)

# Track summary statistics
summary_stats = []

for seed_idx in range(1, CONFIG['NUM_SEEDS'] + 1):
    current_seed = CONFIG['BASE_RANDOM_STATE'] + seed_idx

    log_print("\n" + "=" * 80)
    log_print(f"SEED {seed_idx}/{CONFIG['NUM_SEEDS']} (Random State: {current_seed})")
    log_print("=" * 80)

    strategy = CONFIG['SAMPLING_STRATEGY']
    if strategy == 1:
        meta_sampled = strategy_1_balanced_gender(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                  current_seed, CONFIG['GENDER_COLUMN'])
    elif strategy == 2:
        meta_sampled = strategy_2_stratified_proportional(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                          current_seed, CONFIG['GENDER_COLUMN'])
    elif strategy == 3:
        meta_sampled = strategy_3_random_sampling(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                  current_seed, CONFIG['GENDER_COLUMN'])
    elif strategy == 4:
        meta_sampled = strategy_4_pathology_stratified(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                       current_seed, CONFIG['GENDER_COLUMN'])
    elif strategy == 5:
        meta_sampled = strategy_5_pathology_stratified_clean(meta_unique, CONFIG['SAMPLE_SIZE'],
                                                             current_seed, CONFIG['GENDER_COLUMN'])
    else:
        raise ValueError(f"Invalid SAMPLING_STRATEGY: {strategy}. Must be 1, 2, 3, 4, or 5.")

    filenames_to_load = set(meta_sampled['filename'].values)
    log_print(f"Loading {len(filenames_to_load):,} embeddings...")

    emb_df = load_embeddings_for_filenames(filenames_to_load)

    log_print(f"Matched {len(emb_df):,} embeddings ({len(emb_df)/len(filenames_to_load)*100:.2f}%)")

    if len(emb_df) == 0:
        log_print(f"WARNING: No embeddings matched for seed {seed_idx}!")
        continue

    df_final = pd.merge(meta_sampled, emb_df, on="filename", how="inner")

    # Save as PICKLE to preserve multi-dimensional embeddings!
    output_filename = f"{output_folder}_seed{seed_idx}.pkl"
    output_path = os.path.join(output_dir, output_filename)

    df_final.to_pickle(output_path)

    file_size_mb = os.path.getsize(output_path) / (1024*1024)
    log_print(f"Saved: {output_filename} ({len(df_final):,} rows, {file_size_mb:.2f} MB)")

    # Track stats for summary
    gender_dist = df_final[CONFIG['GENDER_COLUMN']].value_counts().to_dict()
    summary_stats.append({
        'seed': seed_idx,
        'random_state': current_seed,
        'samples': len(df_final),
        'file_size_mb': file_size_mb,
        **{f'gender_{k}': v for k, v in gender_dist.items()}
    })

    # Verify embedding shape on first seed
    if seed_idx == 1:
        log_print(f"Embedding shape preserved: {df_final['embedding'].iloc[0].shape}")
        log_print(f"Embedding size: {df_final['embedding'].iloc[0].flatten().shape[0]:,} features")

    del emb_df, df_final, meta_sampled
    gc.collect()

# Save summary CSV
log_print("\n" + "=" * 80)
log_print("SAVING SUMMARY REPORT")
log_print("=" * 80)

summary_df = pd.DataFrame(summary_stats)
summary_path = os.path.join(output_dir, f"summary_{output_folder}.csv")
summary_df.to_csv(summary_path, index=False)
log_print(f"Summary saved to: {summary_path}")

log_print("\n" + "=" * 80)
log_print("ALL SEEDS PROCESSED SUCCESSFULLY!")
log_print("=" * 80)
log_print(f"\nOutput directory: {output_dir}")
log_print(f"Total files created: {CONFIG['NUM_SEEDS']} pickle files")
log_print(f"Format: PICKLE (full multi-dimensional embeddings preserved!)")
log_print(f"\nFiles generated:")
log_print(f"  - {CONFIG['NUM_SEEDS']} data files: {output_folder}_seed1.pkl ... {output_folder}_seed{CONFIG['NUM_SEEDS']}.pkl")
log_print(f"  - Summary: summary_{output_folder}.csv")
log_print(f"  - Log: {log_filename}")
log_print("=" * 80)

# Save log file
save_log()

CHEST X-RAY GENDER CLASSIFICATION PIPELINE - MULTIPLE SEEDS
Timestamp: 2025-12-13 09:14:18
Target Sample Size: 2,000 samples
Number of Seeds: 20
Base Random State: 42
Train/Test split: 80% / 20%
Sampling Strategy: Balanced Gender (50/50)
Output Directory: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned-seeding/data_type1
Output Format: PICKLE (preserves multi-dimensional embeddings!)

STEP 1: LOADING COMPLETE METADATA
Loaded metadata from: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/verified_dicom_clincalinfo.csv
TOTAL ROWS IN FULL DATASET: 243,334
Original columns: ['dicom_path', 'dicom_id', 'subject_id', 'study_id', 'PerformedProcedureStepDescription', 'ViewPosition', 'Rows', 'Columns', 'ProcedureCodeSequence_CodeMeaning', 'ViewCodeSequence_CodeMeaning']...

Creating filename column from dicom_path...
Stripped file extensions from filenames to match NPZ format
Created filename column from dicom_path
Rows after removing invalid paths: 243,334

Loading embeddings:  98%|█████████▊| 48/49 [24:44<00:30, 30.92s/it]


Matched 2,000 embeddings (100.00%)
Saved: data_type1_seed1.pkl (2,000 rows, 673.14 MB)
Embedding shape preserved: (8, 8, 1376)
Embedding size: 88,064 features

SEED 2/20 (Random State: 44)

STRATEGY 1 - BALANCED GENDER (50/50) - Seed 44
Target: 2,000 samples (1,000 per gender)
Sample size: 2,000 (F=1,000 | M=1,000)
Loading 2,000 embeddings...


Loading embeddings:  98%|█████████▊| 48/49 [22:53<00:28, 28.61s/it]


Matched 2,000 embeddings (100.00%)
Saved: data_type1_seed2.pkl (2,000 rows, 673.14 MB)

SEED 3/20 (Random State: 45)

STRATEGY 1 - BALANCED GENDER (50/50) - Seed 45
Target: 2,000 samples (1,000 per gender)
Sample size: 2,000 (F=1,000 | M=1,000)
Loading 2,000 embeddings...


...

Matched 2,000 embeddings (100.00%)
Saved: data_type1_seed18.pkl (2,000 rows, 673.14 MB)

SEED 19/20 (Random State: 61)

STRATEGY 1 - BALANCED GENDER (50/50) - Seed 61
Target: 2,000 samples (1,000 per gender)
Sample size: 2,000 (F=1,000 | M=1,000)
Loading 2,000 embeddings...


Loading embeddings:  98%|█████████▊| 48/49 [21:49<00:27, 27.29s/it]


Matched 2,000 embeddings (100.00%)
Saved: data_type1_seed19.pkl (2,000 rows, 673.14 MB)

SEED 20/20 (Random State: 62)

STRATEGY 1 - BALANCED GENDER (50/50) - Seed 62
Target: 2,000 samples (1,000 per gender)
Sample size: 2,000 (F=1,000 | M=1,000)
Loading 2,000 embeddings...


Loading embeddings:  98%|█████████▊| 48/49 [23:23<00:29, 29.25s/it]


Matched 2,000 embeddings (100.00%)
Saved: data_type1_seed20.pkl (2,000 rows, 673.14 MB)

SAVING SUMMARY REPORT
Summary saved to: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned-seeding/data_type1/summary_data_type1.csv

ALL SEEDS PROCESSED SUCCESSFULLY!

Output directory: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned-seeding/data_type1
Total files created: 20 pickle files
Format: PICKLE (full multi-dimensional embeddings preserved!)

Files generated:
  - 20 data files: data_type1_seed1.pkl ... data_type1_seed20.pkl
  - Summary: summary_data_type1.csv
  - Log: preprocessing_log_multipleseeds_data_type1_20251213_091418.txt

Log saved to: /content/drive/MyDrive/projects/Quantum-Echos/mimic-cxs_data/data-cleaned-seeding/data_type1/preprocessing_log_multipleseeds_data_type1_20251213_091418.txt
